# Program Pengujian: Mengenali 1 Gambar

Alur: *upload* 1 gambar, baca berkas bobot, hitung umpan maju saja (tanpa koreksi bobot), lalu tampilkan apakah gambar dikenali atau tidak.
Mengikuti *Application procedure* Fausett, *Fundamentals of Neural Networks* (1994), hlm. 299 sampai 300, dengan notasi dan cara penulisan yang sama seperti `TRAINING.ipynb`.

Jalankan sel dari atas ke bawah. Sel 4 akan meminta Anda memilih 1 gambar. Sel 5 sampai 7 memperlihatkan tiap langkah dengan angka dimasukkan ke rumus, lalu Sel 8 menampilkan hasilnya.

## Sel 1. Pengaturan Lingkungan dan Ambang Batas

* Menghubungkan Google Drive, memuat daftar 3 nama kelas, serta menetapkan *threshold* (batas nilai minimal agar gambar dinyatakan dikenali) sebesar 0.5. Ambang ini pilihan kita, bukan bagian dari buku.
* **Analogi:** tim penyortir foto yang sudah dilatih disiapkan di ruang kerja. Mereka membawa buku catatan hasil latihan (`BOBOT.npz`) dan satu aturan: yang dianggap dikenali hanya jawaban yang cukup yakin.

In [ ]:
import io                      # membaca berkas dari memori (bukan dari disk)
import os                      # akses folder dan berkas
import numpy as np             # operasi matriks dan vektor
from PIL import Image          # membaca gambar (Pillow)
import matplotlib.pyplot as plt  # menggambar grafik dan gambar
from google.colab import drive, files  # drive: sambung ke Google Drive; files: dialog unggah berkas
from IPython.display import display, Math   # Math: menampilkan persamaan LaTeX

drive.mount('/content/drive')  # sambungkan Colab ke Google Drive

BERKAS_BOBOT = '/content/drive/MyDrive/TUGAS_ALGORITMA_AI/BOBOT/BOBOT.npz'   # hasil dari program training
AMBANG_DIKENALI = 0.5    # threshold: nilai output terbesar minimal 0,5 agar gambar dianggap dikenali
NAMA_KELAS = ['PESAWAT', 'MOBIL', 'KAPAL']   # urutan sama dengan node output Y1, Y2, Y3

# Opsional: isi bila Anda tahu kelas gambar yang akan diunggah (0 = pesawat, 1 = mobil, 2 = kapal)
# agar hasil dibandingkan dengan kelas sebenarnya. Biarkan None bila tidak ingin dibandingkan.
KELAS_SEBENARNYA = None

print('=' * 52)
print('PENGATURAN')
print('=' * 52)
print(f'Berkas bobot    : {BERKAS_BOBOT}')
print(f'Ambang dikenali : {AMBANG_DIKENALI}')
print(f'Kelas           : {", ".join(NAMA_KELAS)}')

## Sel 2. Step 0: Pemuatan Bobot Hasil Pelatihan

* Membaca berkas `BOBOT.npz` dari Drive (atau menyediakan dialog unggah berkas bila belum tersedia) dan memvalidasi kecocokan ukuran jaringan ($1024 \times 100 \times 3$).
* Di buku, langkah awal prosedur ini memakai bobot hasil *training*, bukan bobot acak.
* **Analogi:** tim membuka buku catatan hasil latihan: tingkat percaya dan kecenderungan yang sudah dipelajari.

In [ ]:
if os.path.exists(BERKAS_BOBOT):
    bobot = np.load(BERKAS_BOBOT)            # baca berkas .npz hasil training
    sumber = 'Google Drive'
else:
    print('Berkas bobot tidak ada di Drive. Silakan unggah BOBOT.npz.')
    unggahan = files.upload()                # files.upload: dialog Colab untuk memilih berkas dari komputer
    # list(unggahan.keys())[0]: nama berkas pertama yang diunggah; io.BytesIO: bungkus isinya agar bisa dibaca np.load
    bobot = np.load(io.BytesIO(unggahan[list(unggahan.keys())[0]]))
    sumber = 'unggahan'

v   = bobot['v']      # bobot V: dari node input ke node hidden
v_0 = bobot['v_0']    # bias node hidden
w   = bobot['w']      # bobot W: dari node hidden ke node output
w_0 = bobot['w_0']    # bias node output
LEBAR, TINGGI = [int(a) for a in bobot['ukuran']]   # ukuran gambar yang dipakai saat training

n = v.shape[0]   # node input: satu node untuk tiap pixel
p = v.shape[1]   # node hidden: lapisan tengah pengolah pola
m = w.shape[1]   # node output: satu per kelas

print('=' * 62)
print(f'Step 0 : Bobot hasil training dibaca (dari {sumber})')
print('=' * 62)
print(f'{"Simbol buku":<13}{"Nama di kode":<14}{"Ukuran":<14}{"Jumlah nilai":>12}')
for simbol, nama, larik in [('v_ij', 'v', v), ('v_0j', 'v_0', v_0), ('w_jk', 'w', w), ('w_0k', 'w_0', w_0)]:
    print(f'{simbol:<13}{nama:<14}{str(larik.shape):<14}{larik.size:>12,}')
print(f'\nNode input (n) : {n}  ({LEBAR} x {TINGGI} pixel)')
print(f'Node hidden (p): {p}')
print(f'Node output (m): {m}')
# assert: hentikan program bila ukuran bobot tidak sesuai (harus 3 node output dan n = lebar x tinggi)
assert m == 3 and n == LEBAR * TINGGI, 'Ukuran bobot tidak sesuai'

## Sel 3. Fungsi Aktivasi dan Prapemrosesan

* Menyiapkan fungsi aktivasi sigmoid $f(x)$ dan fungsi `praproses` yang mengubah gambar menjadi vektor bernilai 0 sampai 1 (langkah sama persis dengan program *training*).
* **Analogi:** foto baru disiapkan dengan cara yang sama seperti kartu foto latihan: diubah menjadi abu-abu, dikecilkan bila perlu, lalu tiap titik diberi nilai terang 0 sampai 1.

In [ ]:
def f(x):
    """Fungsi aktivasi sigmoid biner: f(x) = 1 / (1 + e^-x)."""
    return 1.0 / (1.0 + np.exp(-x))


def praproses(gambar):
    """Ubah satu gambar PIL menjadi vektor x berisi n nilai, rentang 0 sampai 1 (sama dengan training)."""
    abu = gambar.convert('L')                # grayscale: satu nilai terang per pixel
    if abu.size != (LEBAR, TINGGI):          # pengaman ukuran
        abu = abu.resize((LEBAR, TINGGI))
    return np.asarray(abu, dtype=np.float64).flatten() / 255.0   # flatten, lalu bagi 255

print('Fungsi siap: f(x) dan praproses(gambar)')

## Sel 4. Pengunggahan Citra Uji

* Menampilkan tombol pemilih berkas untuk memasukkan satu gambar yang ingin diuji, lalu membaca ukuran dan mode warnanya (RGB = berwarna, L = abu-abu).
* **Analogi:** satu foto baru diserahkan kepada tim untuk ditebak.

In [ ]:
unggahan = files.upload()                      # dialog Colab: pilih 1 gambar dari komputer
nama_berkas = list(unggahan.keys())[0]         # nama berkas pertama yang diunggah
if len(unggahan) > 1:
    print(f'Anda memilih {len(unggahan)} berkas. Yang dipakai hanya yang pertama: {nama_berkas}')

gambar = Image.open(io.BytesIO(unggahan[nama_berkas]))   # io.BytesIO: bungkus isi berkas agar bisa dibaca Pillow
print(f'Berkas   : {nama_berkas}')
print(f'Ukuran   : {gambar.size[0]} x {gambar.size[1]} pixel')
print(f'Mode     : {gambar.mode}')

## Sel 5. Step 2: *Input Unit* Menerima $x_i$

$$x_i = \frac{\text{nilai pixel ke-}i}{255}$$

* Step 1 di buku: untuk setiap vektor *input*, kerjakan Step 2 sampai 4. Di program ini hanya ada 1 gambar uji, jadi Step 2 sampai 4 dikerjakan sekali.
* Gambar diubah oleh `praproses` menjadi vektor `x` berisi 1024 nilai (satu per *node input*).
* **Analogi:** 1024 pengamat masing-masing menerima satu titik terang dari foto, lalu meneruskannya kepada para ahli.

In [ ]:
x = praproses(gambar)     # x: 1024 nilai pixel ternormalisasi (x_1 sampai x_1024)

print('=' * 62)
print('Step 1 : For each input vector, do Steps 2 to 4  (di sini 1 gambar uji)')
print('Step 2 : Input unit X_i menerima x_i')
print('=' * 62)
display(Math(r'x_i = \frac{\text{nilai pixel ke-}i}{255}'))
print(f'Gambar {LEBAR} x {TINGGI} pixel menjadi {len(x)} nilai (x_1 sampai x_{len(x)}).')
print(f'Rentang nilai x : {x.min():.4f} sampai {x.max():.4f}  (0 = hitam, 1 = putih)')
print(f'Contoh          : x_1 = {x[0]:.4f}, x_2 = {x[1]:.4f}, x_3 = {x[2]:.4f}, ..., x_{len(x)} = {x[-1]:.4f}')

## Sel 6. Step 3: Hitung Keluaran *Hidden Layer*

$$z\_in_j = v_{0j} + \sum_{i=1}^{n} x_i v_{ij}, \qquad z_j = f(z\_in_j)$$

* Fungsi `umpan_maju_hidden` menghitung tiap *node hidden* $j$ dengan *loop*, tanpa `@`. Setara `langkah_4` di `TRAINING.ipynb`.
* Output memperlihatkan pengerjaan lengkap untuk $j = 1$, lalu hasil seluruh 100 *node hidden*.
* **Analogi:** tiap ahli menjumlahkan pendapat 1024 pengamat (dikali tingkat percayanya), menambah kecenderungan pribadinya, lalu menyimpulkan seberapa aktif ia.

In [ ]:
def umpan_maju_hidden(x, v, v_0):
    """Step 3: z_in_j = v_0j + SUM_i x_i v_ij, lalu z_j = f(z_in_j)."""
    z_in = np.zeros(p)
    z    = np.zeros(p)
    for j in range(p):
        z_in[j] = v_0[j] + np.sum(x * v[:, j])   # np.sum: jumlahkan semua suku i
        z[j]    = f(z_in[j])
    return z_in, z


z_in, z = umpan_maju_hidden(x, v, v_0)

print('=' * 62)
print('Step 3 : Hitung keluaran hidden layer')
print('=' * 62)
display(Math(r'z\_in_j = v_{0j} + \sum_{i=1}^{n} x_i v_{ij} \qquad z_j = f(z\_in_j) = \frac{1}{1 + e^{-z\_in_j}}'))
print('Keterangan: tiap baris tabel = satu node hidden (j = 1 sampai 100), n = 1024 pixel')
print('  z_in (masukan bersih): hasil penjumlahan, belum menjadi keluaran. Nilainya bisa berapa saja')
print('  z (keluaran node hidden): z = f(z_in), selalu 0 sampai 1, dikirim ke node output')
print('  z dekat 1 berarti node hidden aktif, dekat 0 berarti tidak aktif')
print()
print('Pengerjaan untuk node hidden 1 (j = 1):')
suku = x * v[:, 0]                       # suku ke-i: x_i x v_i1
jumlah = np.sum(suku)                    # jumlah 1024 suku
display(Math(r'z\_in_1 = v_{01} + x_1 v_{1,1} + x_2 v_{2,1} + x_3 v_{3,1} + \dots + x_{1024} v_{1024,1}'))
display(Math(rf'z\_in_1 = {v_0[0]:.4f} + ({x[0]:.4f})({v[0, 0]:.4f}) + ({x[1]:.4f})({v[1, 0]:.4f}) + ({x[2]:.4f})({v[2, 0]:.4f}) + \dots + ({x[-1]:.4f})({v[-1, 0]:.4f})'))
display(Math(rf'z\_in_1 = {v_0[0]:.4f} + ({suku[0]:.4f}) + ({suku[1]:.4f}) + ({suku[2]:.4f}) + \dots + ({suku[-1]:.4f})'))
display(Math(rf'z\_in_1 = {v_0[0]:.4f} + ({jumlah:.4f}) = {z_in[0]:.4f}'))
display(Math(rf'z_1 = f({z_in[0]:.4f}) = \frac{{1}}{{1 + e^{{-({z_in[0]:.4f})}}}} = \frac{{1}}{{1 + {np.exp(-z_in[0]):.4f}}} = {z[0]:.4f}'))
print()
print('Hasil untuk seluruh node hidden:')
print(f'{"j":>4}{"z_in (masukan)":>17}{"z (keluaran)":>15}')
for j in range(p):
    print(f'{j + 1:>4}{z_in[j]:>+17.4f}{z[j]:>15.4f}')

## Sel 7. Step 4: Hitung Keluaran *Output Layer*

$$y\_in_k = w_{0k} + \sum_{j=1}^{p} z_j w_{jk}, \qquad y_k = f(y\_in_k)$$

* Fungsi `umpan_maju_output` menghitung tiap *node output* $k$ dengan *loop*, tanpa `@`. Setara `langkah_5` di `TRAINING.ipynb`.
* Output memperlihatkan pengerjaan lengkap untuk ketiga *node output*. Angka $z_1$ sampai $z_{100}$ sama untuk ketiganya (hasil Sel 6), yang berbeda hanya bobot $w_{jk}$ dan *bias* $w_{0k}$.
* **Analogi:** tiga juru bicara (pesawat, mobil, kapal) masing-masing menjumlahkan pendapat 100 ahli (dikali tingkat percayanya), lalu menyatakan seberapa yakin foto itu kelasnya.

In [ ]:
def umpan_maju_output(z, w, w_0):
    """Step 4: y_in_k = w_0k + SUM_j z_j w_jk, lalu y_k = f(y_in_k)."""
    y_in = np.zeros(m)
    y    = np.zeros(m)
    for k in range(m):
        y_in[k] = w_0[k] + np.sum(z * w[:, k])
        y[k]    = f(y_in[k])
    return y_in, y


y_in, y = umpan_maju_output(z, w, w_0)

print('=' * 62)
print('Step 4 : Hitung keluaran output layer')
print('=' * 62)
display(Math(r'y\_in_k = w_{0k} + \sum_{j=1}^{p} z_j w_{jk} \qquad y_k = f(y\_in_k)'))
print('Keterangan: tiap baris = satu node output (satu kelas), p = 100 node hidden')
print('  y_in (masukan bersih): hasil penjumlahan, belum menjadi keluaran')
print('  y (keluaran node output): y = f(y_in), jawaban jaringan, dekat 1 berarti jaringan memilih kelas ini')
print()
for k in range(m):
    suku = z * w[:, k]                   # suku ke-j: z_j x w_jk
    jumlah = np.sum(suku)                # jumlah 100 suku
    print(f'Pengerjaan untuk node output {k + 1} (k = {k + 1}, kelas {NAMA_KELAS[k]}):')
    display(Math(rf'y\_in_{k + 1} = w_{{0,{k + 1}}} + z_1 w_{{1,{k + 1}}} + z_2 w_{{2,{k + 1}}} + z_3 w_{{3,{k + 1}}} + \dots + z_{{100}} w_{{100,{k + 1}}}'))
    display(Math(rf'y\_in_{k + 1} = {w_0[k]:.4f} + ({z[0]:.4f})({w[0, k]:.4f}) + ({z[1]:.4f})({w[1, k]:.4f}) + ({z[2]:.4f})({w[2, k]:.4f}) + \dots + ({z[-1]:.4f})({w[-1, k]:.4f})'))
    display(Math(rf'y\_in_{k + 1} = {w_0[k]:.4f} + ({suku[0]:.4f}) + ({suku[1]:.4f}) + ({suku[2]:.4f}) + \dots + ({suku[-1]:.4f})'))
    display(Math(rf'y\_in_{k + 1} = {w_0[k]:.4f} + ({jumlah:.4f}) = {y_in[k]:.4f}'))
    display(Math(rf'y_{k + 1} = f({y_in[k]:.4f}) = \frac{{1}}{{1 + e^{{-({y_in[k]:.4f})}}}} = \frac{{1}}{{1 + {np.exp(-y_in[k]):.4f}}} = {y[k]:.4f}'))
    print()
print('Rangkuman ketiga node output:')
print(f'{"k":>4}{"Kelas":>10}{"y_in (masukan)":>17}{"y (keluaran)":>15}')
for k in range(m):
    print(f'{k + 1:>4}{NAMA_KELAS[k]:>10}{y_in[k]:>+17.4f}{y[k]:>15.4f}')

## Sel 8. Hasil: Klasifikasi dan Visualisasi

* Memilih kelas dengan nilai keluaran $y_k$ tertinggi, lalu menentukan status pengenalan (bagian ini bukan dari buku, melainkan aturan kita):
   * Jika nilai tertinggi $\ge 0.5$, citra DIKENALI.
   * Jika nilai tertinggi $< 0.5$, citra TIDAK DIKENALI.
* Menampilkan 3 panel: gambar asli, bentuk *grayscale* $32 \times 32$ yang diproses jaringan, serta grafik batang nilai keluaran ($Y_1, Y_2, Y_3$) beserta garis *threshold*.
* **Analogi:** pelatih mengambil jawaban juru bicara yang paling yakin. Bila keyakinannya kurang dari batas, tim mengaku tidak mengenali fotonya.

In [ ]:
# x dan y sudah dihitung di Sel 5 sampai 7
indeks = int(np.argmax(y))          # argmax: nomor node output bernilai terbesar = kelas tebakan
nilai_maks = float(y[indeks])
dikenali = nilai_maks >= AMBANG_DIKENALI   # True bila nilai terbesar mencapai threshold

print('=' * 52)
print('HASIL PENGENALAN')
print('=' * 52)
print(f'{"Node output":<14}{"Kelas":<10}{"Nilai":>8}')
for k in range(m):
    tanda = '  <-- terbesar' if k == indeks else ''
    print(f'Y{k + 1:<13}{NAMA_KELAS[k]:<10}{y[k]:>8.3f}{tanda}')
print('-' * 52)

if dikenali:
    print(f'DIKENALI sebagai {NAMA_KELAS[indeks]} (nilai {nilai_maks:.3f}, ambang {AMBANG_DIKENALI})')
else:
    print(f'TIDAK DIKENALI (nilai terbesar {nilai_maks:.3f}, di bawah ambang {AMBANG_DIKENALI})')

if KELAS_SEBENARNYA is not None:
    cocok = (indeks == KELAS_SEBENARNYA) and dikenali
    print(f'Kelas sebenarnya: {NAMA_KELAS[KELAS_SEBENARNYA]} -> {"BENAR" if cocok else "SALAH"}')

# 3 panel: gambar asli, gambar yang masuk jaringan, grafik batang nilai output
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(11, 3.2))
a1.imshow(gambar); a1.set_title('Gambar asli'); a1.axis('off')
a2.imshow(x.reshape(TINGGI, LEBAR), cmap='gray', vmin=0, vmax=1); a2.set_title('Masuk ke jaringan (grayscale)'); a2.axis('off')
# batang hijau = node terbesar, abu-abu = lainnya; garis merah putus-putus = threshold
warna = ['tab:green' if k == indeks else 'tab:gray' for k in range(m)]
a3.bar([f'Y{k + 1}\n{NAMA_KELAS[k]}' for k in range(m)], y, color=warna)
a3.axhline(AMBANG_DIKENALI, color='tab:red', linestyle='--', linewidth=1)
a3.set_ylim(0, 1); a3.set_title('Nilai output')
plt.tight_layout()
plt.show()